# S6E5 — Feature Engineering

## 1. Imports e configuração

In [ ]:
from pathlib import Path

from mltemplate.config import ProjectConfig
from mltemplate.storage import StorageManager
from mltemplate.data import KaggleSource, DataManager
from mltemplate.features import FeatureEngineer

import logging
logging.basicConfig(level=logging.INFO)

In [ ]:
config = ProjectConfig(
    target="PitNextLap",
    numerical_features=[],    # preencher com base no 01_eda.ipynb
    categorical_features=[],  # preencher com base no 01_eda.ipynb
    ignore_features=["id"],
    problem_type="regression",
)

storage = StorageManager(root=Path("."))
dm      = DataManager(storage, config)

## 2. Carregar dados

In [ ]:
source = KaggleSource("playground-series-s6e5")
train_df, test_df = dm.load_raw(source)

print(f"Train: {train_df.shape}")
print(f"Test:  {test_df.shape}")

## 3. Split treino / validação

In [ ]:
X_train, X_val, y_train, y_val = dm.split(train_df)

print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")

## 4. Pipeline de transformação

In [ ]:
fe = FeatureEngineer(config)

# 4.1 Remover colunas ignoradas
X_train = fe.drop_ignored(X_train)
X_val   = fe.drop_ignored(X_val)
X_test  = fe.drop_ignored(test_df)

# 4.2 Imputação (mediana para numéricas, moda para categóricas)
X_train = fe.impute(X_train)
X_val   = fe.impute(X_val)
X_test  = fe.impute(X_test)

# 4.3 Encoding categórico (OneHot, fit apenas no treino)
fe.fit_encoder(X_train)
X_train = fe.encode(X_train)
X_val   = fe.encode(X_val)
X_test  = fe.encode(X_test)

# 4.4 Scaling numérico (StandardScaler, fit apenas no treino)
fe.fit_scaler(X_train)
X_train = fe.scale(X_train)
X_val   = fe.scale(X_val)
X_test  = fe.scale(X_test)

In [ ]:
# [Opcional] Features polinomiais
# fe.fit_poly(X_train, degree=2)
# X_train = fe.expand_poly(X_train)
# X_val   = fe.expand_poly(X_val)
# X_test  = fe.expand_poly(X_test)

## 5. Inspeção do resultado

In [ ]:
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")
X_train.head()

## 6. Salvar feature set

In [ ]:
dm.save_feature_set(X_train, X_test, name="v1")
print("Feature set 'v1' salvo.")